# Data Preparation: Standardizing Column Names
Here we load the raw CSV files, standardize their column names (lowercase, remove special characters, replace spaces with underscores), and save the cleaned versions to `data/02_processed/`.

In [277]:
import pandas as pd
import numpy as np

In [278]:
billings = pd.read_csv("../../data/01_raw/raw_billings.csv")
billings.shape
billings.info()
billings.head()

C:\Users\NuluShreya\AppData\Local\Temp\ipykernel_25332\1029342165.py:1: DtypeWarning: Columns (0: Proforma_Auto_Renewal, 1: Proforma_World_Pay_Token, 2: Current_Anchor_List, 3: Last_Renewal, 4: Last_Band) have mixed types. Specify dtype option on import or set low_memory=False.
  billings = pd.read_csv("../../data/01_raw/raw_billings.csv")


<class 'pandas.DataFrame'>
RangeIndex: 122082 entries, 0 to 122081
Data columns (total 59 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   Co_Ref                      122082 non-null  str    
 1   Renewal_Month               122082 non-null  str    
 2   Connection_Net              8037 non-null    float64
 3   Connection_Qty              8037 non-null    float64
 4   Discount_Amount             13551 non-null   str    
 5   Sustainability_Score        122082 non-null  float64
 6   Total_Renewal_Score_New     122082 non-null  float64
 7   Starting_Connection_Net     8545 non-null    float64
 8   Starting_Connection_Qty     8545 non-null    float64
 9   Last_Years_Price            112992 non-null  float64
 10  Last_Years_Date_Paid        0 non-null       float64
 11  Auto_Renewal_Score          122082 non-null  int64  
 12  Status_Scores               122082 non-null  int64  
 13  Anchoring_Score          

,Co_Ref,Renewal_Month,Connection_Net,Connection_Qty,Discount_Amount,Sustainability_Score,Total_Renewal_Score_New,Starting_Connection_Net,Starting_Connection_Qty,Last_Years_Price,...,Connection_Group,Tenure_Group,#_of_Connection,Last_Renewal,Last_Band,Last_Total_Net_Paid,Last_Connections,Anchor_Group,Renewal_Year,DateTime_Out
0,VT6174,01-11-2024,NaN,NaN,NaN,8.0,42.5,NaN,NaN,799.0,...,1,3,1.0,01-11-2023,Band B,664.0,1.0,1,2024,01-11-2024
1,VD3828,01-08-2025,NaN,NaN,NaN,8.0,41.5,NaN,NaN,799.0,...,1,1,1.0,NaN,NaN,NaN,NaN,1,2025,01-08-2025
2,DV8120,01-03-2025,NaN,NaN,NaN,8.0,33.0,NaN,NaN,799.0,...,1,4+,1.0,01-03-2024,Band C1,749.0,1.0,1,2025,01-03-2025
3,EZ9894,01-06-2025,NaN,NaN,NaN,9.5,44.5,NaN,NaN,799.0,...,1,4+,1.0,01-06-2024,Band C1,749.0,1.0,1,2025,01-06-2025
4,FA8957,01-03-2025,NaN,NaN,NaN,9.5,42.5,NaN,NaN,799.0,...,1,3,1.0,01-03-2024,Band C1,749.0,1.0,1,2025,01-03-2025


In [279]:
billings = billings.drop_duplicates()
print("Shape after duplicate removal:", billings.shape)

Shape after duplicate removal: (122082, 59)


In [280]:
billings.columns = (
    billings.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

In [281]:
date_cols = [
    "renewal_month",
    "proforma_date",
    "registration_date",
    "prospect_renewal_date",
    "closed_date",
    "last_renewal",
    "datetime_out"
]

for col in date_cols:
    if col in billings.columns:
        billings[col] = pd.to_datetime(
            billings[col],
            errors="coerce"
        )

In [260]:
obj_cols = billings.select_dtypes(include="object").columns

for col in obj_cols:
    billings[col] = billings[col].astype(str).str.strip()

C:\Users\NuluShreya\AppData\Local\Temp\ipykernel_25332\2747719410.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  obj_cols = billings.select_dtypes(include="object").columns


In [261]:
# check missing percentage
missing_pct = billings.isnull().mean() * 100
missing_summary = missing_pct.sort_values(ascending=False)
print("Missing values (%) by column:")
print(missing_summary.head(65))

Missing values (%) by column:
last_years_date_paid          100.000000
connection_qty                 93.416720
connection_net                 93.416720
starting_connection_net        93.000606
starting_connection_qty        93.000606
discount_amount                88.900084
closed_date                    62.747989
registration_date              62.100064
prospect_renewal_date          61.545519
last_connections               39.629102
last_total_net_paid            39.609443
last_band                      39.572582
last_renewal                   39.556200
current_anchor_list            21.261939
total_net_paid                 17.083600
payment_timeframe              17.083600
proforma_world_pay_token       14.786783
proforma_auto_renewal          14.786783
proforma_date                  13.653118
proforma_audit_status           7.559673
proforma_account_stage          7.559673
last_years_price                7.445815
tenure_years                    0.833866
tenure_group               

In [262]:
# columns with >80% missing values
high_missing_cols = missing_pct[missing_pct > 80].index

print(f"\nDropping {len(high_missing_cols)} columns with >80% missing:")
print(list(high_missing_cols))

billings = billings.drop(columns=high_missing_cols)


Dropping 6 columns with >80% missing:
['connection_net', 'connection_qty', 'discount_amount', 'starting_connection_net', 'starting_connection_qty', 'last_years_date_paid']


In [263]:
billings.dtypes

co_ref                                   str
renewal_month                 datetime64[us]
sustainability_score                 float64
total_renewal_score_new              float64
last_years_price                     float64
auto_renewal_score                     int64
status_scores                          int64
anchoring_score                      float64
tenure_scores                        float64
proforma_auto_renewal                    str
proforma_world_pay_token                 str
proforma_date                 datetime64[us]
current_anchorings                     int64
current_anchor_list                      str
payment_timeframe                    float64
registration_date             datetime64[us]
proforma_account_stage                   str
proforma_audit_status                    str
current_auto_renewal_flag                str
current_world_pay_token                  str
renewal_score_at_release             float64
proforma_membership_status               str
proforma_a

In [264]:
numeric_cols = [
    "tenure_years",
    "amount",
    "total_amount",
    "gross",
    "membership_net",
    "package_net",
    "pqqnet",
    "total_net_paid",
    "starting_net",
    "starting_gross" 
]

for col in numeric_cols:
    if col in billings.columns:
        billings[col] = pd.to_numeric(
            billings[col],
            errors="coerce"
        )

In [265]:
# historical columns
hist_num = ["last_connections", "last_total_net_paid", "last_years_price"]
for col in hist_num:
    billings[col] = billings[col].fillna(0)

hist_cat = ["last_band", "last_renewal"]
for col in hist_cat:
    billings[col] = billings[col].fillna("No_History")

# payment columns
billings["payment_timeframe"] = billings["payment_timeframe"].fillna("Unknown")
billings["total_net_paid"] = billings["total_net_paid"].fillna(0)
billings["proforma_auto_renewal"] = billings["proforma_auto_renewal"].fillna("FALSE")
billings["proforma_world_pay_token"] = billings["proforma_world_pay_token"].fillna("FALSE")

# workflow columns
billings["proforma_account_stage"] = billings["proforma_account_stage"].fillna("Not_Started")
billings["proforma_audit_status"] = billings["proforma_audit_status"].fillna("Not_Started")


In [266]:
# moderate missing business columns
billings["current_anchor_list"] = billings["current_anchor_list"].fillna("Unknown")


# very low missing numerical columns
billings["tenure_years"] = billings["tenure_years"].fillna(
    billings["tenure_years"].median()
)

billings["renewal_score_at_release"] = billings["renewal_score_at_release"].fillna(
    billings["renewal_score_at_release"].median()
)

# very low missing categorical columns
low_cat_cols = [
    "tenure_group",
    "proforma_membership_status",
    "anchor_group",
    "connection_group",
    "band"
]

for col in low_cat_cols:
    billings[col] = billings[col].fillna("Unknown")

# connection count
billings["#_of_connection"] = billings["#_of_connection"].fillna(0)

# approved list
billings["proforma_approved_lists"] = billings["proforma_approved_lists"].fillna("FALSE")

date_cols_with_missing = [
    "registration_date",
    "prospect_renewal_date",
    "proforma_date"
]

for col in date_cols_with_missing:
    billings[col + "_missing_flag"] = billings[col].isnull().astype(int)

In [267]:
print("🔍 Final Data Quality Check")

# 1) remaining nulls
print("\nRemaining nulls:")
print(billings.isnull().sum().sort_values(ascending=False).head(10))

# 2) negative financial values
financial_cols = [
    "amount", "total_amount", "gross",
    "membership_net", "package_net",
    "total_net_paid", "starting_net", "starting_gross"
]

for col in financial_cols:
    if col in billings.columns:
        neg_count = (billings[col] < 0).sum()
        if neg_count > 0:
            print(f"{col}: {neg_count} negative values")

🔍 Final Data Quality Check

Remaining nulls:
closed_date              76604
registration_date        75813
prospect_renewal_date    75136
proforma_date            16668
last_years_price             0
auto_renewal_score           0
co_ref                       0
renewal_month                0
anchoring_score              0
status_scores                0
dtype: int64


In [268]:
# check missing percentage
missing_pct = billings.isnull().mean() * 100
missing_summary = missing_pct.sort_values(ascending=False)
print("Missing values (%) by column:")
print(missing_summary.head(65))

Missing values (%) by column:
closed_date                           62.747989
registration_date                     62.100064
prospect_renewal_date                 61.545519
proforma_date                         13.653118
last_years_price                       0.000000
auto_renewal_score                     0.000000
co_ref                                 0.000000
renewal_month                          0.000000
anchoring_score                        0.000000
status_scores                          0.000000
proforma_auto_renewal                  0.000000
tenure_scores                          0.000000
proforma_world_pay_token               0.000000
current_anchorings                     0.000000
current_anchor_list                    0.000000
payment_timeframe                      0.000000
proforma_account_stage                 0.000000
proforma_audit_status                  0.000000
current_auto_renewal_flag              0.000000
current_world_pay_token                0.000000
renewal_sc

In [269]:
billings.isnull().sum().sort_values(ascending=False).head(20)

closed_date                  76604
registration_date            75813
prospect_renewal_date        75136
proforma_date                16668
last_years_price                 0
auto_renewal_score               0
co_ref                           0
renewal_month                    0
anchoring_score                  0
status_scores                    0
proforma_auto_renewal            0
tenure_scores                    0
proforma_world_pay_token         0
current_anchorings               0
current_anchor_list              0
payment_timeframe                0
proforma_account_stage           0
proforma_audit_status            0
current_auto_renewal_flag        0
current_world_pay_token          0
dtype: int64

In [270]:
billings.to_csv("../../data/02_processed/processed_billings.csv", index=False)
print("Processed billings saved!")

Processed billings saved!


In [271]:
print("Final shape:", billings.shape)
display(billings.head())

Final shape: (122082, 56)


,co_ref,renewal_month,sustainability_score,total_renewal_score_new,last_years_price,auto_renewal_score,status_scores,anchoring_score,tenure_scores,proforma_auto_renewal,...,last_renewal,last_band,last_total_net_paid,last_connections,anchor_group,renewal_year,datetime_out,registration_date_missing_flag,prospect_renewal_date_missing_flag,proforma_date_missing_flag
0,VT6174,2024-01-11,8.0,42.5,799.0,9,9,7.5,9.0,True,...,2023-01-11 00:00:00,Band B,664.0,1.0,1,2024,2024-01-11,0,0,0
1,VD3828,2025-01-08,8.0,41.5,799.0,9,9,7.5,8.0,True,...,No_History,No_History,0.0,0.0,1,2025,2025-01-08,0,0,0
2,DV8120,2025-01-03,8.0,33.0,799.0,8,0,7.5,9.5,True,...,2024-01-03 00:00:00,Band C1,749.0,1.0,1,2025,2025-01-03,0,0,0
3,EZ9894,2025-01-06,9.5,44.5,799.0,9,9,7.5,9.5,True,...,2024-01-06 00:00:00,Band C1,749.0,1.0,1,2025,2025-01-06,1,1,1
4,FA8957,2025-01-03,9.5,42.5,799.0,9,8,7.5,8.5,True,...,2024-01-03 00:00:00,Band C1,749.0,1.0,1,2025,2025-01-03,1,1,0


In [272]:
# # 🔍 EXPLORATION: Remaining Data Quality Issues
# print("="*60)
# print("REMAINING DATA QUALITY CHECKS")
# print("="*60)

# # 1. Check for "Unknown" over-representation in categorical columns
# print("\n1️⃣  CATEGORICAL COLUMNS - 'Unknown' Fill Rate:")
# for col in billings.select_dtypes(include="object").columns:
#     unknown_pct = (billings[col] == "Unknown").sum() / len(billings) * 100
#     if unknown_pct > 5:
#         print(f"   {col}: {unknown_pct:.1f}% Unknown")

# # 2. Check for negative values in financial columns
# print("\n2️⃣  NEGATIVE VALUES in Financial Columns:")
# financial_cols = ["amount", "total_amount", "gross", "membership_net", "package_net", 
#                   "total_net_paid", "starting_net", "starting_gross", "discount_amount"]
# has_negatives = False
# for col in financial_cols:
#     if col in billings.columns:
#         neg_count = (billings[col] < 0).sum()
#         if neg_count > 0:
#             print(f"   {col}: {neg_count} negative values")
#             has_negatives = True
# if not has_negatives:
#     print("   ✅ None found")

# # 3. Check date column issues
# print("\n3️⃣  DATE COLUMNS - NaT (Missing) Percentage:")
# date_cols_check = ["last_years_date_paid", "proforma_date", "registration_date", 
#                    "prospect_renewal_date", "closed_date"]
# for col in date_cols_check:
#     if col in billings.columns:
#         nat_pct = billings[col].isna().sum() / len(billings) * 100
#         print(f"   {col}: {nat_pct:.1f}% NaT")

# # 4. Check categorical value consistency (top values)
# print("\n4️⃣  CATEGORICAL COLUMNS - Top Values (potential issues):")
# for col in billings.select_dtypes(include="object").columns:
#     top_val = billings[col].value_counts().head(1).index[0]
#     top_pct = billings[col].value_counts().head(1).values[0] / len(billings) * 100
#     if top_pct > 50:
#         print(f"   {col}: '{top_val}' = {top_pct:.1f}% (⚠️ high concentration)")
#     else:
#         print(f"   {col}: {billings[col].nunique()} unique values")

# # 5. Zero values in financial columns
# print("\n5️⃣  ZERO VALUES in Financial Columns (potential data issues):")
# for col in financial_cols:
#     if col in billings.columns:
#         zero_count = (billings[col] == 0).sum()
#         zero_pct = zero_count / len(billings) * 100
#         if zero_count > len(billings) * 0.1:  # More than 10%
#             print(f"   {col}: {zero_count} zeros ({zero_pct:.1f}%)")

# print("\n" + "="*60)

In [273]:
# # 🔍 EXPLORATION: Remaining Data Quality Issues
# print("="*60)
# print("REMAINING DATA QUALITY CHECKS")
# print("="*60)

# # 1. Check for "Unknown" over-representation in categorical columns
# print("\n1️⃣  CATEGORICAL COLUMNS - 'Unknown' Fill Rate:")
# for col in billings.select_dtypes(include="object").columns:
#     unknown_pct = (billings[col] == "Unknown").sum() / len(billings) * 100
#     if unknown_pct > 5:
#         print(f"   {col}: {unknown_pct:.1f}% Unknown")

# # 2. Check for negative values in financial columns
# print("\n2️⃣  NEGATIVE VALUES in Financial Columns:")
# financial_cols = ["amount", "total_amount", "gross", "membership_net", "package_net", 
#                   "total_net_paid", "starting_net", "starting_gross", "discount_amount"]
# has_negatives = False
# for col in financial_cols:
#     if col in billings.columns:
#         neg_count = (billings[col] < 0).sum()
#         if neg_count > 0:
#             print(f"   {col}: {neg_count} negative values")
#             has_negatives = True
# if not has_negatives:
#     print("   ✅ None found")

# # 3. Check date column issues
# print("\n3️⃣  DATE COLUMNS - NaT (Missing) Percentage:")
# date_cols_check = ["last_years_date_paid", "proforma_date", "registration_date", 
#                    "prospect_renewal_date", "closed_date"]
# for col in date_cols_check:
#     if col in billings.columns:
#         nat_pct = billings[col].isna().sum() / len(billings) * 100
#         print(f"   {col}: {nat_pct:.1f}% NaT")

# # 4. Check categorical value consistency (top values)
# print("\n4️⃣  CATEGORICAL COLUMNS - Top Values (potential issues):")
# for col in billings.select_dtypes(include="object").columns:
#     top_val = billings[col].value_counts().head(1).index[0]
#     top_pct = billings[col].value_counts().head(1).values[0] / len(billings) * 100
#     if top_pct > 50:
#         print(f"   {col}: '{top_val}' = {top_pct:.1f}% (⚠️ high concentration)")
#     else:
#         print(f"   {col}: {billings[col].nunique()} unique values")

# # 5. Zero values in financial columns
# print("\n5️⃣  ZERO VALUES in Financial Columns (potential data issues):")
# for col in financial_cols:
#     if col in billings.columns:
#         zero_count = (billings[col] == 0).sum()
#         zero_pct = zero_count / len(billings) * 100
#         if zero_count > len(billings) * 0.1:  # More than 10%
#             print(f"   {col}: {zero_count} zeros ({zero_pct:.1f}%)")

# print("\n" + "="*60)

In [274]:
# # 🔄 FINAL NULL/UNKNOWN VALUE HANDLING

# print("\n📋 FINAL NULL & UNKNOWN VALUE CLEANUP...")

# # Re-run null-fill operations after dropping columns
# print(f"   Before final cleanup: {billings.isnull().sum().sum()} nulls")

# # Categorical columns: Replace "Unknown" with mode or a specific value
# cat_cols_final = billings.select_dtypes(include="object").columns
# for col in cat_cols_final:
#     unknown_count = (billings[col] == "Unknown").sum()
#     if unknown_count > 0:
#         # Replace Unknown with the most common non-Unknown value
#         mode_val = billings[billings[col] != "Unknown"][col].mode()
#         if len(mode_val) > 0:
#             billings[col] = billings[col].replace("Unknown", mode_val[0])
#             print(f"   ✅ Replaced Unknown in '{col}' with '{mode_val[0]}'")

# # Numeric columns: Fill remaining NaN with median
# num_cols_final = billings.select_dtypes(include=np.number).columns
# for col in num_cols_final:
#     if billings[col].isnull().sum() > 0:
#         billings[col] = billings[col].fillna(billings[col].median())

# print(f"   After final cleanup: {billings.isnull().sum().sum()} nulls")
# print("✅ Data is now clean and ready for modeling!")

In [275]:

# # Quick verification of remaining nulls
# null_cols = billings.isnull().sum()
# print("\n📊 REMAINING NULLS BY COLUMN:")
# print(null_cols[null_cols > 0].sort_values(ascending=False).head(10))
# print(f"\nTotal rows: {len(billings)}")
# print(f"Total nulls: {null_cols.sum()}")

In [276]:

# # 🗑️ DROP COMPLETELY NULL COLUMNS & FINAL HANDLING

# print("\n🗑️  FINAL DATA CLEANUP - Remove Useless Columns")

# # Drop columns that are 100% null
# for col in billings.columns:
#     if billings[col].isnull().sum() == len(billings):
#         billings = billings.drop(columns=[col])
#         print(f"   ✅ Dropped '{col}' (100% null)")

# # Handle proforma_date (has some NaT) - fill with a default date or drop
# if "proforma_date" in billings.columns:
#     nat_pct = billings["proforma_date"].isnull().sum() / len(billings) * 100
#     # Fill NaT dates with median date
#     billings["proforma_date"] = billings["proforma_date"].fillna(
#         billings["proforma_date"].median()
#     )
#     print(f"   ✅ Filled '{nat_pct:.1f}% NaT in 'proforma_date' with median")

# print(f"\n✅ FINAL DATA STATUS:")
# print(f"   Shape: {billings.shape}")
# print(f"   Total nulls: {billings.isnull().sum().sum()}")
# print(f"   Memory usage: {billings.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# # Save the cleaned data
# billings.to_csv("../../data/02_processed/processed_billings.csv", index=False)
# print(f"   ✅ Saved to processed_billings.csv")